# Audio Feature Extraction (openSMILE) — MELD

Extractor: [openSMILE](https://github.com/audeering/opensmile-python) — hand-crafted acoustic functionals  
Feature set: configurable via `FEATURE_SET` in the config cell (default: `eGeMAPSv02`)  
Feature level: `Functionals` — one fixed-size vector per utterance  
Output: `Dataset/Processed/MELD/features/audio_opensmile_{FEATURE_TAG}_{split}.pt`  
Format: `dict {clip_name: np.array(N_FEATURES,)}` per split  

| Feature set | Dim | Notes |
|---|---|---|
| `eGeMAPSv02` | 88 | Geneva Minimalistic Acoustic Parameter Set v2 — standard for SER |
| `GeMAPSv01b` | 62 | Older GeMAPS baseline |
| `ComParE_2016` | 6373 | Large set used in ComParE challenges |
| `IS09` | 384 | Interspeech 2009 emotion challenge set |
| `IS10` | 1582 | Interspeech 2010 paralinguistics challenge set |
| `IS13` | 6373 | Interspeech 2013 ComParE set |

No GPU needed — CPU-only extraction.

In [1]:
import opensmile
import numpy as np
import pandas as pd
import torch
import torchaudio
from pathlib import Path
from tqdm.notebook import tqdm

In [ ]:
PROJECT_ROOT = Path("/mnt/Work/ML/Thesis/BMVC/Hopeful")
DATA_ROOT    = PROJECT_ROOT / "Dataset/Processed/MELD"
FEATURES_DIR = DATA_ROOT / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000
SPLITS      = ["train", "dev", "test"]

# ── Swap feature set here ─────────────────────────────────────────────────────
FEATURE_SET = opensmile.FeatureSet.IS10
# Options:
#   opensmile.FeatureSet.eGeMAPSv02    → 88 features  (recommended for SER)
#   opensmile.FeatureSet.GeMAPSv01b   → 62 features
#   opensmile.FeatureSet.ComParE_2016 → 6373 features
#   opensmile.FeatureSet.IS09         → 384 features  (Interspeech 2009 emotion)
#   opensmile.FeatureSet.IS10         → 1582 features (Interspeech 2010 paralinguistics)
#   opensmile.FeatureSet.IS13         → 6373 features (Interspeech 2013 ComParE)
# ─────────────────────────────────────────────────────────────────────────────
FEATURE_TAG = FEATURE_SET.name  # e.g. "eGeMAPSv02"

print(f"Feature set : {FEATURE_SET.name}")
print(f"Tag         : {FEATURE_TAG}")
print(f"Data root   : {DATA_ROOT}")

Feature set : IS10
Tag         : IS10
Data root   : /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD


In [3]:
smile = opensmile.Smile(
    feature_set=FEATURE_SET,
    feature_level=opensmile.FeatureLevel.Functionals,
    num_workers=1,
    verbose=False,
)

N_FEATURES = smile.num_features
print(f"Dimensions  : {N_FEATURES}")
print(f"Feature names (first 10): {smile.feature_names[:10]}")

Dimensions  : 1582
Feature names (first 10): ['pcm_loudness_sma_maxPos', 'pcm_loudness_sma_minPos', 'pcm_loudness_sma_amean', 'pcm_loudness_sma_linregc1', 'pcm_loudness_sma_linregc2', 'pcm_loudness_sma_linregerrA', 'pcm_loudness_sma_linregerrQ', 'pcm_loudness_sma_stddev', 'pcm_loudness_sma_skewness', 'pcm_loudness_sma_kurtosis']


In [4]:
labels = pd.read_csv(DATA_ROOT / "labels.csv")
print(f"Total utterances : {len(labels)}")
print(f"Split counts:")
print(labels["split"].value_counts())

Total utterances : 13708
Split counts:
split
train    9989
test     2610
dev      1109
Name: count, dtype: int64


In [5]:
def load_and_resample(path: Path) -> np.ndarray:
    """Load wav, resample to SAMPLE_RATE, return mono float32 numpy."""
    wav, sr = torchaudio.load(str(path))
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    return wav.mean(0).numpy()  # (T,)


def extract_features(path: Path) -> np.ndarray:
    """Extract openSMILE functionals from one audio file → (N_FEATURES,)."""
    df = smile.process_file(str(path))
    return df.values[0].astype(np.float32)  # (N_FEATURES,)

In [6]:
all_skipped: dict[str, list[str]] = {}

for split in SPLITS:
    split_df   = labels[labels["split"] == split].reset_index(drop=True)
    clip_names = split_df["clip_name"].tolist()
    audio_paths = split_df["audio_path"].tolist()
    print(f"\n--- {split} ({len(split_df)} utterances) ---")

    features: dict[str, np.ndarray] = {}
    skipped:  list[str] = []

    for cname, apath in tqdm(zip(clip_names, audio_paths), total=len(clip_names), desc=split):
        p = DATA_ROOT / apath
        if not p.exists():
            skipped.append(cname)
            continue
        try:
            features[cname] = extract_features(p)
        except Exception as e:
            print(f"  [warn] {cname}: {e}")
            skipped.append(cname)

    out_path = FEATURES_DIR / f"audio_opensmile_{FEATURE_TAG}_{split}.pt"
    torch.save(features, out_path)
    size_mb = out_path.stat().st_size / 1e6
    print(f"Saved   : {out_path}  ({size_mb:.1f} MB, {len(features)} entries)")
    if skipped:
        print(f"Skipped : {skipped}")
    all_skipped[split] = skipped


--- train (9989 utterances) ---


train:   0%|          | 0/9989 [00:00<?, ?it/s]

/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/opensmile/core/smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/opensmile/core/smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/opensmile/core/smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/opensmile/core/smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/opensmile/core/smi

Saved   : /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD/features/audio_opensmile_IS10_train.pt  (89.2 MB, 9988 entries)
Skipped : ['dia125_utt3']

--- dev (1109 utterances) ---


dev:   0%|          | 0/1109 [00:00<?, ?it/s]

/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/opensmile/core/smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))
/mnt/Work/Environments/Ubuntu/Conda/envs/hopeful/lib/python3.10/site-packages/opensmile/core/smile.py:297: UserWarning: Segment too short, filling with NaN.
  warnings.warn(UserWarning("Segment too short, filling with NaN."))


Saved   : /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD/features/audio_opensmile_IS10_dev.pt  (9.9 MB, 1108 entries)
Skipped : ['dia110_utt7']

--- test (2610 utterances) ---


test:   0%|          | 0/2610 [00:00<?, ?it/s]

Saved   : /mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/MELD/features/audio_opensmile_IS10_test.pt  (23.3 MB, 2610 entries)


In [7]:
print("=== Verification ===")

for split in SPLITS:
    out_path  = FEATURES_DIR / f"audio_opensmile_{FEATURE_TAG}_{split}.pt"
    loaded    = torch.load(out_path, weights_only=False)
    split_df  = labels[labels["split"] == split]
    n_skipped = len(all_skipped.get(split, []))
    expected  = len(split_df) - n_skipped

    assert len(loaded) == expected, f"{split}: Expected {expected}, got {len(loaded)}"

    sample_key  = next(iter(loaded))
    sample_feat = loaded[sample_key]
    assert sample_feat.shape == (N_FEATURES,), f"{split}: shape {sample_feat.shape}"

    all_feats = np.stack(list(loaded.values()))
    has_nan   = np.isnan(all_feats).any()
    has_inf   = np.isinf(all_feats).any()

    sample_row = labels[labels.clip_name == sample_key].iloc[0]
    print(f"\n[{split}]")
    print(f"  Count   : {len(loaded)} / {len(split_df)} ({n_skipped} skipped)  ✓")
    print(f"  Shape   : {sample_feat.shape}  ✓")
    print(f"  Mean    : {all_feats.mean():.4f}")
    print(f"  Std     : {all_feats.std():.4f}")
    print(f"  Has NaN : {has_nan}")
    print(f"  Has Inf : {has_inf}")
    print(f"  Sample  : {sample_key}  |  emotion={sample_row.emotion}")

print("\nAll assertions passed.")

=== Verification ===

[train]
  Count   : 9988 / 9989 (1 skipped)  ✓
  Shape   : (1582,)  ✓
  Mean    : nan
  Std     : nan
  Has NaN : True
  Has Inf : False
  Sample  : dia0_utt0  |  emotion=neutral

[dev]
  Count   : 1108 / 1109 (1 skipped)  ✓
  Shape   : (1582,)  ✓
  Mean    : nan
  Std     : nan
  Has NaN : True
  Has Inf : False
  Sample  : dia0_utt0  |  emotion=neutral

[test]
  Count   : 2610 / 2610 (0 skipped)  ✓
  Shape   : (1582,)  ✓
  Mean    : 27.3599
  Std     : 301.2918
  Has NaN : False
  Has Inf : False
  Sample  : dia0_utt0  |  emotion=neutral

All assertions passed.
